# Web Chatbot penelitian

Aplikasi ini dibangun dengan menggunakan arsitektur sederhana di mana **Google Scholar** (melalui library `scholarly`) bertindak sebagai sumber data pencarian. **Gemini API** bertindak sebagai otak LLM yang mensintesis jawaban dengan `temperature=0` (agar outputnya sangat faktual dan deterministik). **Streamlit** sebagai antarmuka penggunanya. **Ngrok** akan digunakan untuk mengekspos server lokal/Colab ke internet.


### 1. Persiapan Lingkungan (Instalasi Library)

In [15]:
!pip install streamlit google-genai scholarly pyngrok -q

### 2. Script Aplikasi (Simpan sebagai `app.py`)

Script ini berisi logika antarmuka Streamlit, pencarian Google Scholar, dan pemanggilan Gemini API.

In [ ]:
import streamlit as st
import google.generativeai as genai
from scholarly import scholarly

# --- KONFIGURASI HALAMAN STREAMLIT ---
st.set_page_config(page_title="Chatbot Penelitian LLM", page_icon="📚", layout="wide")
st.title("📚 Chatbot Asisten Penelitian")
st.caption("Berbasis Gemini 1.5 & Google Scholar (Temperature = 0)")

# --- SIDEBAR: KONFIGURASI API KEY ---
with st.sidebar:
    st.header("Konfigurasi")
    api_key = st.text_input("Masukkan Google Gemini API Key:", type="password")
    if not api_key:
        st.warning("Silakan masukkan API Key untuk memulai.")

# --- FUNGSI PENCARIAN GOOGLE SCHOLAR ---
@st.cache_data(show_spinner=False)
def search_google_scholar(query, max_results=3):
    """Mencari publikasi di Google Scholar berdasarkan kata kunci."""
    results = []
    try:
        search_query = scholarly.search_pubs(query)
        for i in range(max_results):
            pub = next(search_query)
            title = pub.get('bib', {}).get('title', 'Tidak ada judul')
            abstract = pub.get('bib', {}).get('abstract', 'Tidak ada abstrak')
            author = pub.get('bib', {}).get('author', 'Tidak diketahui')
            year = pub.get('bib', {}).get('pub_year', 'Tahun tidak diketahui')

            results.append(f"Judul: {title}\nPenulis: {author}\nTahun: {year}\nAbstrak: {abstract}")
    except StopIteration:
        pass # Tidak ada lagi hasil yang ditemukan
    except Exception as e:
        return f"Error saat mengambil data dari Scholar: {e}"

    return "\n\n---\n\n".join(results) if results else "Tidak ada literatur yang ditemukan untuk kueri ini."

# --- FUNGSI LLM GEMINI ---
def generate_response(prompt, scholar_context):
    """Menghasilkan respons dari Gemini berdasarkan konteks Scholar."""
    genai.configure(api_key=api_key)

    # Mengatur parameter temperature = 0
    generation_config = genai.types.GenerationConfig(temperature=0.0)

    # Menggunakan model flash untuk kecepatan (bisa diganti ke pro)
    model = genai.GenerativeModel('gemini-1.5-flash', generation_config=generation_config)

    system_prompt = f"""
    Anda adalah asisten peneliti akademik yang sangat cerdas.
    Tugas Anda adalah menjawab pertanyaan pengguna berdasarkan literatur jurnal yang disediakan di bawah ini.
    Jika literatur tidak menjawab pertanyaan, katakan sejujurnya bahwa data tidak mencukupi, jangan mengarang informasi (halusinasi).

    [LITERATUR GOOGLE SCHOLAR]:
    {scholar_context}

    [PERTANYAAN PENGGUNA]:
    {prompt}
    """

    response = model.generate_content(system_prompt)
    return response.text

# --- LOGIKA CHAT INTERFACE ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# Tampilkan riwayat obrolan
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Input dari pengguna
prompt = st.chat_input("Tanyakan sesuatu tentang topik penelitian Anda... (misal 'Apa dampak AI pada pendidikan?')")
if prompt:

    # Tambahkan pertanyaan ke antarmuka
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    if not api_key:
        st.error("API Key belum dimasukkan. Silakan cek sidebar.")
    else:
        with st.chat_message("assistant"):
            with st.spinner("Mencari literatur di Google Scholar..."):
                # Ekstrak kata kunci sederhana (Anda bisa menggunakan LLM juga untuk ekstraksi kata kunci)
                scholar_context = search_google_scholar(prompt)
                st.info("Berdasarkan literatur yang ditemukan:\n" + scholar_context[:300] + "...") # Preview singkat

            with st.spinner("Menyusun sintesis dengan Gemini..."):
                try:
                    answer = generate_response(prompt, scholar_context)
                    st.markdown(answer)
                    # Simpan ke riwayat
                    st.session_state.messages.append({"role": "assistant", "content": answer})
                except Exception as e:
                    st.error(f"Terjadi kesalahan pada Gemini API: {e}")

### 3. Menjalankan Aplikasi dengan Ngrok

Penggunaan `pyngrok` bersama Streamlit digunakan jika menjalankan kode di lingkungan *cloud* (seperti Google Colab) atau ingin membagikan akses localhost Anda ke orang lain.

Buat file baru bernama `run_ngrok.py` (atau jalankan ini di sel berikutnya jika menggunakan Colab):

In [ ]:
from pyngrok import ngrok
import subprocess
import time
from google.colab import userdata

# 1. Masukkan AuthToken Ngrok Anda (Dapatkan dari dashboard ngrok.com)
# NGROK_AUTH_TOKEN = "MASUKKAN_TOKEN_NGROK_ANDA_DI_SINI" # Removed placeholder
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN') # Get token from Colab secrets
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Jalankan Streamlit di background pada port 8501
print("Memulai Streamlit...")
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Tunggu beberapa detik agar server Streamlit menyala
time.sleep(3)

# 3. Buka tunnel Ngrok ke port 8501
public_url = ngrok.connect(8501)
print(f"✅ Aplikasi berhasil dijalankan!")
print(f"🌍 Akses Chatbot Anda melalui link ini: {public_url}")

# Menjaga proses tetap hidup
try:
    process.wait()
except KeyboardInterrupt:
    print("Menutup server dan ngrok...")
    ngrok.kill()

### Cara Kerja Arsitektur Ini:

1. **RAG (Retrieval-Augmented Generation) Sederhana:** Ketika pengguna bertanya, sistem tidak langsung bertanya kepada LLM. Sistem akan mencari *keyword* dari pertanyaan tersebut ke Google Scholar terlebih dahulu.
2. **Konteks:** Ekstraksi judul, penulis, dan abstrak dari jurnal yang relevan lalu digabungkan menjadi satu teks panjang.
3. **`temperature=0`:** Prompt pengguna beserta konteks jurnal dikirimkan ke Gemini API. Karena `temperature` diset ke `0.0`, Gemini akan merespons secara sangat kaku, faktual, dan tidak kreatif, yang mana ini adalah praktik terbaik (Best Practice) untuk meminimalisir halusinasi dalam penelitian.
4. **Caching:** Fungsi `search_google_scholar` dibungkus dengan `@st.cache_data` agar pertanyaan yang sama tidak melakukan *scraping* berulang ke Google Scholar, yang bisa menyebabkan IP Anda diblokir sementara oleh Google.

Untuk menghentikan app dan menjalankan app berikutnya, jalankan cell di bawah:

In [18]:
# Hentikan app sebelumnya sebelum menjalankan app baru
import os, signal
try:
    proc.terminate()
    print("App dihentikan.")
except:
    pass